# Inspect pipeline.py

Notebook di ispezione: stessa sequenza di chiamate di `pipeline.py` (funzioni `run_cleaning1`, `run_cleaning2`, `run_all`), ma una cella per passaggio cosi' si puo' controllare il DataFrame dopo ogni step.

Nessuna logica e' stata modificata: le celle chiamano le funzioni di `pipeline.py` esattamente nell'ordine in cui compaiono negli orchestratori.

Dataset di riferimento: `config.ADNIMERGE` (cambia `cfg = ...` per ispezionare un altro dataset, es. `PTDEMOG`, `PLASMA_PANEL`, `PLASMA_NFL`).

## Setup

In [1]:
import numpy as np
import pandas as pd

import config
from config import DatasetConfig, ADNIMERGE, PTDEMOG
from pipeline import *

cfg = PTDEMOG

## CLEANING 1 — funzioni atomiche (df -> df)

Segue l'ordine di `run_cleaning1()` in pipeline.py.

### load — download_file (ex client.download_file)

In [2]:
df = load(cfg)
df.shape

(6149, 84)

In [3]:
df.head()

,PHASE,PTID,RID,VISCODE,VISCODE2,VISDATE,PTSOURCE,PTGENDER,PTDOB,PTDOBYY,...,PTBIRPR,PTBIRGR,ID,SITEID,USERDATE,USERDATE2,DD_CRF_VERSION_LABEL,LANGUAGE_CODE,HAS_QC_ERROR,update_stamp
0,ADNI1,011_S_0002,2,sc,sc,2005-08-17,1.0,1.0,04/1931,1931-01-01,...,NaN,NaN,18,107,2005-08-17,NaN,NaN,NaN,NaN,2005-08-17 00:00:00
1,ADNI1,022_S_0001,1,f,f,2005-08-18,1.0,2.0,12/1944,1944-01-01,...,NaN,NaN,20,10,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00
2,ADNI1,011_S_0003,3,sc,sc,2005-08-18,1.0,1.0,05/1924,1924-01-01,...,NaN,NaN,22,107,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00
3,ADNI1,022_S_0004,4,sc,sc,2005-08-18,1.0,1.0,01/1938,1938-01-01,...,NaN,NaN,24,10,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00
4,ADNI1,011_S_0005,5,sc,sc,2005-08-23,1.0,1.0,12/1931,1931-01-01,...,NaN,NaN,26,107,2005-08-23,NaN,NaN,NaN,NaN,2005-08-23 00:00:00


### drop_columns — rimuove colonne indesiderate (es. VISCODE in PTDEMOG)

In [4]:
df, dropped_cols, missing_drop_cols = drop_columns(df, cfg.drop_columns)
print("scartate:", dropped_cols)
print("richieste ma assenti:", missing_drop_cols)
df.shape

scartate: ['VISCODE']
richieste ma assenti: []


(6149, 83)

### replace_unknown — replace_unknown_values

In [5]:
df = replace_unknown(df)
df.head()

,PHASE,PTID,RID,VISCODE2,VISDATE,PTSOURCE,PTGENDER,PTDOB,PTDOBYY,PTHAND,...,PTBIRPR,PTBIRGR,ID,SITEID,USERDATE,USERDATE2,DD_CRF_VERSION_LABEL,LANGUAGE_CODE,HAS_QC_ERROR,update_stamp
0,ADNI1,011_S_0002,2,sc,2005-08-17,1.0,1.0,04/1931,1931-01-01,2.0,...,NaN,NaN,18.0,107,2005-08-17,NaN,NaN,NaN,NaN,2005-08-17 00:00:00
1,ADNI1,022_S_0001,1,f,2005-08-18,1.0,2.0,12/1944,1944-01-01,NaN,...,NaN,NaN,20.0,10,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00
2,ADNI1,011_S_0003,3,sc,2005-08-18,1.0,1.0,05/1924,1924-01-01,1.0,...,NaN,NaN,22.0,107,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00
3,ADNI1,022_S_0004,4,sc,2005-08-18,1.0,1.0,01/1938,1938-01-01,1.0,...,NaN,NaN,24.0,10,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00
4,ADNI1,011_S_0005,5,sc,2005-08-23,1.0,1.0,12/1931,1931-01-01,1.0,...,NaN,NaN,26.0,107,2005-08-23,NaN,NaN,NaN,NaN,2005-08-23 00:00:00


### decensor — toglie i simboli di censura (solo se cfg.decensor_biomarkers)

In [6]:
if cfg.decensor_biomarkers:
    df = decensor(df, cfg.decensor_columns or config.columns_in("Biomarker"))
df.shape

(6149, 83)

### drop_if_all_none — 1° drop: colonne essenziali

In [7]:
n_before = len(df)
df = drop_if_all_none(df, cfg.essential_columns)
print("righe rimosse:", n_before - len(df))
df.shape

righe rimosse: 18


(6131, 83)

### drop_if_all_none — 2° drop: DX obbligatoria (also_required)

In [8]:
n_before = len(df)
df = drop_if_all_none(df, cfg.also_required)
print("righe rimosse:", n_before - len(df))
df.shape

righe rimosse: 171


(5960, 83)

### rename_variables — new_variable_names (CATALOG + override per-file)

In [9]:
df = rename_variables(df, cfg)
df.columns.tolist()

['COLPROT',
 'PTID',
 'RID',
 'VISCODE',
 'EXAMDATE',
 'PTSOURCE',
 'GENDER',
 'PTDOB',
 'PTDOBYY',
 'PTHAND',
 'MARRY',
 'EDUCATION',
 'PTWORKHS',
 'PTWORK',
 'PTNOTRT',
 'PTRTYR',
 'PTHOME',
 'PTTLANG',
 'PTPLANG',
 'PTADBEG',
 'PTCOGBEG',
 'DX',
 'ETHNICITY',
 'RACE',
 'PTIDENT',
 'PTORIENT',
 'PTORIENTOT',
 'PTENGSPK',
 'PTNLANG',
 'PTENGSPKAGE',
 'PTCLANG',
 'PTLANGSP',
 'PTLANGWR',
 'PTSPTIM',
 'PTSPOTTIM',
 'PTLANGPR1',
 'PTLANGSP1',
 'PTLANGRD1',
 'PTLANGWR1',
 'PTLANGUN1',
 'PTLANGPR2',
 'PTLANGSP2',
 'PTLANGRD2',
 'PTLANGWR2',
 'PTLANGUN2',
 'PTLANGPR3',
 'PTLANGSP3',
 'PTLANGRD3',
 'PTLANGWR3',
 'PTLANGUN3',
 'PTLANGPR4',
 'PTLANGSP4',
 'PTLANGRD4',
 'PTLANGWR4',
 'PTLANGUN4',
 'PTLANGPR5',
 'PTLANGSP5',
 'PTLANGRD5',
 'PTLANGWR5',
 'PTLANGUN5',
 'PTLANGPR6',
 'PTLANGSP6',
 'PTLANGRD6',
 'PTLANGWR6',
 'PTLANGUN6',
 'PTLANGTTL',
 'PTETHCATH',
 'PTASIAN',
 'PTOPI',
 'PTBORN',
 'PTBIRPL',
 'PTIMMAGE',
 'PTIMMWHY',
 'PTBIRPR',
 'PTBIRGR',
 'ID',
 'SITEID',
 'USERDATE',
 'USERDAT

### parse_dates — to_date_format

In [10]:
[cfg.date_column] + cfg.extra_date_columns

['EXAMDATE', 'PTDOB', 'update_stamp']

In [11]:
df = parse_dates(df, [cfg.date_column] + cfg.extra_date_columns)
df.dtypes

c:\Users\ChiaraPollicini\OneDrive - Net Service S.p.A\Documenti\Github\DataCleaning\File Benedetta\pipeline.py:96: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[c] = pd.to_datetime(df[c], errors="coerce")


COLPROT                         object
PTID                            object
RID                              int64
VISCODE                         object
EXAMDATE                datetime64[ns]
                             ...      
USERDATE2                       object
DD_CRF_VERSION_LABEL            object
LANGUAGE_CODE                   object
HAS_QC_ERROR                   float64
update_stamp            datetime64[ns]
Length: 83, dtype: object

In [12]:
df.shape

(5960, 83)

### dedup_visits — find_exam_code (elimina visite duplicate, tiene la piu' completa)

In [13]:
n_before = len(df)
df = dedup_visits(df, cfg.id_column, cfg.date_column, cfg.essential_columns)
print("righe rimosse:", n_before - len(df))
df.shape

righe rimosse: 0


(5960, 83)

### add_visit_month — find_exam_code -> VISIT_MONTH

In [14]:
df = add_visit_month(df, cfg.id_column, cfg.date_column)
df[[cfg.id_column, cfg.date_column, "VISIT_MONTH"]].head()

,RID,EXAMDATE,VISIT_MONTH
1,1,2005-08-18,0
0,2,2005-08-17,0
1571,2,2010-09-22,61
2404,2,2011-09-19,73
2,3,2005-08-18,0


### recompute_age — add_calculated_age (solo se cfg.recompute_age)

In [15]:
if cfg.recompute_age:
    df = recompute_age(df, cfg.date_column)
df.filter(regex="AGE").head()

  recompute_age: calcolo di AGE per visita
  recompute_age: calcolo diretto da PTDOB e EXAMDATE


,PTENGSPKAGE,PTIMMAGE,LANGUAGE_CODE,AGE
1,NaN,NaN,NaN,60.711841
0,NaN,NaN,NaN,74.379192
1571,NaN,NaN,NaN,79.477070
2404,NaN,NaN,NaN,80.468172
2,NaN,NaN,NaN,81.297741


In [16]:
df.shape


(5960, 85)

### recode — categorize_* (stringa -> codice numerico, config.RECODE)

In [17]:
df = recode(df, cfg.recode_columns)
df[[c for c in cfg.recode_columns if c in df.columns]].head()

,GENDER,MARRY,ETHNICITY,RACE
1,0,1.0,NaN,NaN
0,1,1.0,0.0,5.0
1571,1,3.0,0.0,5.0
2404,1,3.0,0.0,5.0
2,1,1.0,0.0,5.0


### clean_fs_fields — FLDSTRENG/FSVERSION .str.extract (solo se cfg.clean_fs_fields)

In [18]:
if cfg.clean_fs_fields:
    df = clean_fs_fields(df)
df.filter(regex="FLDSTRENG|FSVERSION").head()

""
1
0
1571
2404
2


### add_constant_columns — stampa colonne-costante (armonizzazione, es. METHOD_PLASMA)

In [19]:
df = add_constant_columns(df, cfg.constant_columns)
df.shape

(5960, 85)

### add_derived_ratios — rapporti generici (es. AB42/AB40)

In [20]:
df = add_derived_ratios(df, cfg.derived_ratios)
df.shape

(5960, 85)

### add_atn_profile — get_ATN_profile (solo se cfg.compute_atn; ADNIMERGE non lo usa)

In [21]:
if cfg.compute_atn:
    df = add_atn_profile(df, cfg)
df.shape

(5960, 85)

### df1 — risultato di cleaning 1 (equivalente a run_cleaning1(cfg))

In [22]:
df1 = df
df1.shape

(5960, 85)

In [23]:
df1.head()

,COLPROT,PTID,RID,VISCODE,EXAMDATE,PTSOURCE,GENDER,PTDOB,PTDOBYY,PTHAND,...,ID,SITEID,USERDATE,USERDATE2,DD_CRF_VERSION_LABEL,LANGUAGE_CODE,HAS_QC_ERROR,update_stamp,VISIT_MONTH,AGE
1,ADNI1,022_S_0001,1,f,2005-08-18,1.0,0,1944-12-01,1944-01-01,NaN,...,20.0,10,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00,0,60.711841
0,ADNI1,011_S_0002,2,sc,2005-08-17,1.0,1,1931-04-01,1931-01-01,2.0,...,18.0,107,2005-08-17,NaN,NaN,NaN,NaN,2005-08-17 00:00:00,0,74.379192
1571,ADNIGO,011_S_0002,2,sc,2010-09-22,1.0,1,1931-04-01,1931-01-01,2.0,...,304.0,8,2010-09-22,NaN,NaN,NaN,NaN,2013-03-22 15:23:58,61,79.477070
2404,ADNI2,011_S_0002,2,m72,2011-09-19,1.0,1,1931-04-01,1931-01-01,2.0,...,636.0,8,2011-09-20,NaN,NaN,NaN,NaN,2013-05-30 10:05:05,73,80.468172
2,ADNI1,011_S_0003,3,sc,2005-08-18,1.0,1,1924-05-01,1924-01-01,1.0,...,22.0,107,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00,0,81.297741


## CLEANING 2 / 3 — ex notebook adni_cleaning2 / adni_cleaning3

Segue l'ordine di `run_cleaning2(df, cfg)` in pipeline.py. Si riparte da `df1`.

In [24]:
df = df1.copy()
log = {}
r0, c0 = df.shape

### drop_sparse_columns — remove_param_few_subjects (solo se cfg.drop_sparse_columns)

In [25]:
if cfg.drop_sparse_columns:
    df, dropped = drop_sparse_columns(df)
    log["colonne_scartate_sparse"] = dropped
df.shape

(5960, 85)

### remove_single_visit_subjects — remove_sub_1visit (solo se cfg.remove_single_visit)

In [26]:
if cfg.remove_single_visit:
    n_before = len(df)
    df, info = remove_single_visit_subjects(df, cfg.id_column, force=cfg.keep_even_single_visit)
    log["single_visit"] = {**info, "righe_rimosse": n_before - len(df)}
df.shape

(5960, 85)

### make_dummies — classes_to_dummies (solo se cfg.make_dummies)

In [27]:
created = []
if cfg.make_dummies:
    df, created = make_dummies(df, cfg.dummy_columns)
    log["dummy_create"] = created
print("dummy create:", created)
df.shape

dummy create: ['GENDER_0', 'GENDER_1', 'MARRY_0', 'MARRY_1', 'MARRY_2', 'MARRY_3', 'ETHNICITY_0', 'ETHNICITY_1', 'RACE_0', 'RACE_1', 'RACE_2', 'RACE_3', 'RACE_4', 'RACE_5']


(5960, 95)

### drop_if_all_none — drop_if_all_none (volumi, riuso di cleaning 1)

In [28]:
if cfg.volume_row_keys:
    n_before = len(df)
    df = drop_if_all_none(df, cfg.volume_row_keys)
    log["righe_rimosse_volumi"] = n_before - len(df)
df.shape

(5960, 95)

### normalize_volumes_icv — transform_volumes_as_ICV_percent (solo se cfg.normalize_icv)

In [29]:
if cfg.normalize_icv:
    df = normalize_volumes_icv(df, cfg.icv_column)
    log["icv_normalizzato"] = True
df.shape

(5960, 95)

### keep_only_columns — filtro colonne finali (whitelist)

In [30]:
if cfg.keep_columns:
    engineered = [c for c in ("VISIT_MONTH", "AGE_bl") if c in df.columns]
    df, dropped_cols, missing_cols = keep_only_columns(df, cfg.keep_columns, extra=created + engineered)
    log["colonne_scartate_finali"] = dropped_cols
    log["colonne_richieste_assenti"] = missing_cols
df.shape

keep_only_columns: keep=['PHASE', 'PTID', 'RID', 'VISCODE', 'EXAMDATE', 'GENDER', 'PTDOB', 'MARRY', 'EDUCATION', 'ETHNICITY', 'RACE', 'PTADBEG', 'PTCOGBEG', 'DX', 'HAS_QC_ERROR', 'update_stamp'], extra=['GENDER_0', 'GENDER_1', 'MARRY_0', 'MARRY_1', 'MARRY_2', 'MARRY_3', 'ETHNICITY_0', 'ETHNICITY_1', 'RACE_0', 'RACE_1', 'RACE_2', 'RACE_3', 'RACE_4', 'RACE_5', 'VISIT_MONTH']


(5960, 26)

In [31]:
missing_cols

['PHASE', 'GENDER', 'MARRY', 'ETHNICITY', 'RACE']

### df2, log — risultato di cleaning 2 (equivalente a run_cleaning2(df1, cfg))

In [32]:
log["shape_iniziale"] = (r0, c0)
log["shape_finale"] = df.shape
df2 = df
log

{'dummy_create': ['GENDER_0',
  'GENDER_1',
  'MARRY_0',
  'MARRY_1',
  'MARRY_2',
  'MARRY_3',
  'ETHNICITY_0',
  'ETHNICITY_1',
  'RACE_0',
  'RACE_1',
  'RACE_2',
  'RACE_3',
  'RACE_4',
  'RACE_5'],
 'colonne_scartate_finali': ['COLPROT',
  'PTSOURCE',
  'PTDOBYY',
  'PTHAND',
  'PTWORKHS',
  'PTWORK',
  'PTNOTRT',
  'PTRTYR',
  'PTHOME',
  'PTTLANG',
  'PTPLANG',
  'PTIDENT',
  'PTORIENT',
  'PTORIENTOT',
  'PTENGSPK',
  'PTNLANG',
  'PTENGSPKAGE',
  'PTCLANG',
  'PTLANGSP',
  'PTLANGWR',
  'PTSPTIM',
  'PTSPOTTIM',
  'PTLANGPR1',
  'PTLANGSP1',
  'PTLANGRD1',
  'PTLANGWR1',
  'PTLANGUN1',
  'PTLANGPR2',
  'PTLANGSP2',
  'PTLANGRD2',
  'PTLANGWR2',
  'PTLANGUN2',
  'PTLANGPR3',
  'PTLANGSP3',
  'PTLANGRD3',
  'PTLANGWR3',
  'PTLANGUN3',
  'PTLANGPR4',
  'PTLANGSP4',
  'PTLANGRD4',
  'PTLANGWR4',
  'PTLANGUN4',
  'PTLANGPR5',
  'PTLANGSP5',
  'PTLANGRD5',
  'PTLANGWR5',
  'PTLANGUN5',
  'PTLANGPR6',
  'PTLANGSP6',
  'PTLANGRD6',
  'PTLANGWR6',
  'PTLANGUN6',
  'PTLANGTTL',
  'PTETH

In [33]:
df2

,PTID,RID,VISCODE,EXAMDATE,PTDOB,EDUCATION,PTADBEG,PTCOGBEG,DX,HAS_QC_ERROR,...,MARRY_3,ETHNICITY_0,ETHNICITY_1,RACE_0,RACE_1,RACE_2,RACE_3,RACE_4,RACE_5,VISIT_MONTH
1,022_S_0001,1,f,2005-08-18,1944-12-01,18.0,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
0,011_S_0002,2,sc,2005-08-17,1931-04-01,16.0,NaN,NaN,NaN,NaN,...,0,1,0,0,0,0,0,0,1,0
1571,011_S_0002,2,sc,2010-09-22,1931-04-01,16.0,NaN,NaN,NaN,NaN,...,1,1,0,0,0,0,0,0,1,61
2404,011_S_0002,2,m72,2011-09-19,1931-04-01,16.0,NaN,NaN,NaN,NaN,...,1,1,0,0,0,0,0,0,1,73
2,011_S_0003,3,sc,2005-08-18,1924-05-01,18.0,1999.0,NaN,NaN,NaN,...,0,1,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6049,404_S_10891,10891,sc,2025-07-03,1951-03-01,20.0,NaN,2019.0,NaN,0.0,...,0,1,0,0,0,0,0,0,1,0
6052,941_S_10893,10893,sc,2025-07-03,1948-12-01,12.0,NaN,2024.0,NaN,0.0,...,1,1,0,0,0,0,0,0,1,0
6057,404_S_10894,10894,sc,2025-07-08,1945-12-01,14.0,NaN,2021.0,NaN,0.0,...,1,1,0,0,0,0,0,0,1,0
6063,404_S_10895,10895,sc,2025-07-09,1958-03-01,16.0,NaN,2015.0,NaN,0.0,...,0,1,0,0,0,0,0,0,1,0


In [34]:
df1['AGE']

1       60.711841
0       74.379192
1571    79.477070
2404    80.468172
2       81.297741
          ...    
6049    74.340862
6052    76.585900
6057    79.600274
6063    67.356605
6072    81.538672
Name: AGE, Length: 5960, dtype: float64

## MERGE per categoria (v0)

Motore generico guidato da `config.CATEGORY_MERGE`. Esempio: la categoria `plasma` ha 2 file registrati (`PLASMA_PANEL`, `PLASMA_NFL`) — servono entrambi puliti (cleaning 1 + 2) prima del merge.

In [ ]:
from config import PLASMA_PANEL, PLASMA_NFL

In [ ]:
# run_cleaning(cfg) restituisce (df, log): prendiamo il df pulito (cleaning1 + cleaning2)
datasets = {
    PLASMA_PANEL.file_code: run_cleaning(PLASMA_PANEL)[0],
    PLASMA_NFL.file_code: run_cleaning(PLASMA_NFL)[0],
}
merged, merge_log = merge_category(datasets, "plasma")
merge_log

In [ ]:
merged.head()

## REPORT

Rigenera cio' che prima era l'Excel `_statistics` (output, non input). Report su `df1` (cleaned 1), come in `run_all()`.

In [ ]:
report = profile(df1, cfg.cohort_column)
report.head(20)

## SAVE

Unico punto (insieme al blocco `__main__`) che tocca il disco. Celle NON eseguite di default: eseguile solo se vuoi davvero sovrascrivere i CSV in `cfg.output_cleaned1` / `cfg.output_cleaned2` / `cfg.report_file`.

In [ ]:
# save_dataset(df1, cfg.output_cleaned1)
# report.to_csv(cfg.report_file, index=False)
# save_dataset(df2, cfg.output_cleaned2)

## run_all — orchestratore completo (tutti i file registrati in config.DATASETS)

Cella NON eseguita di default: `run_all()` pulisce tutti i dataset registrati, mergia le categorie e scrive tutti i CSV su disco (come `python pipeline.py`).

In [ ]:
# summary = run_all()
# summary